# Short Term Trading Strategies That Work
## Alpaca data → strategy signals → vectorbt trades → performance matrix

**How to use:** run Setup once, enter your credentials when prompted, set ticker(s) and dates, then run any strategy section. Each section has its own input cell. Use **Run All** to compare all documented strategies in this book. You do not need to edit YAML or package code.

**Local:** extract the project ZIP, open this notebook inside `notebooks/`, and use Python 3.12. **Colab:** upload this notebook, then upload the project ZIP when Setup asks. The same notebook works in both environments.

[Strategy reference](https://docs.google.com/document/d/1aIOvM_YQpMVRlTOkTYumFFm5qzOdUQizo5ubrBOvaws/edit). Source-close results are idealized; next-open is an explicitly different timing assumption. This notebook defaults to real Alpaca data, never synthetic fallback.

### Contents
1. 2-Period RSI Under 5
2. Cumulative RSI — Two-Day Sum
3. Double 7’s
4. VIX RSI
5. TRIN
6. S&P Short
7. Ten-Day Closing Low — Five-Day Hold
8. Overnight SPY Holding Test

## 0. Setup — install the shared project

Run this cell once per fresh runtime. If NumPy/pandas were imported before installation and changed versions, restart the runtime and rerun Setup. Only Python 3.12 has been runtime-tested. The code lives in the shared package; these notebooks contain usage examples, not duplicated trading logic.

In [ ]:
from pathlib import Path
import sys, subprocess, importlib.util, zipfile, io

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT = Path("/content/larryconnors-trading")
    if not (PROJECT / "pyproject.toml").exists():
        from google.colab import files
        print("Upload the provided larryconnors-trading.zip project archive.")
        uploaded = files.upload()
        archives = [name for name in uploaded if name.endswith(".zip")]
        if len(archives) != 1:
            raise ValueError("Upload exactly one project ZIP")
        with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
            for name in archive.namelist():
                path = Path(name)
                if path.is_absolute() or ".." in path.parts or path.parts[0] != "larryconnors-trading":
                    raise ValueError("Unexpected archive layout")
            archive.extractall("/content")
        del uploaded
else:
    PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists() and (p / "src/connors").exists()), None)
    if PROJECT is None:
        raise RuntimeError("Open this notebook from inside the extracted project folder")

assert (3, 11) <= sys.version_info[:2] < (3, 14), "Use Python 3.12 (tested)"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                       "-c", str(PROJECT / "constraints.txt"), "-e", str(PROJECT) + "[dev]"])
sys.path.insert(0, str(PROJECT / "src"))
print("Project ready:", PROJECT)


### Alpaca credentials and imports

Set **API_KEY** and **API_SECRET** in your local environment or in Colab **Secrets** (enable notebook access). Otherwise, hidden input prompts appear. Keys stay in memory and are never printed or written into reports. No trading orders are submitted.

The package uses `StockHistoricalDataClient.get_stock_bars(StockBarsRequest(..., timeframe=TimeFrame.Day))`. SDK pagination retrieves all pages; the adapter caches bars across strategies and downloads warm-up history before your start date.

In [ ]:
# The official Alpaca SDK supplies historical data; vectorbt runs the portfolios.
from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame
from connors.alpaca_sdk import read_notebook_secret
from connors.notebook import NotebookResearch, performance_matrix, display_performance_matrix
from connors.notebook_strategies import (
    PositionType, high_low_strategy, rsi_25_75_strategy, r3_strategy,
    percent_b_strategy, multiple_days_strategy, rsi_sma_strategy, tps_strategy,
    rsi_under_5_strategy, cumulative_rsi_strategy, double_seven_strategy,
    vix_rsi_strategy, trin_strategy, sp_short_strategy, ten_day_low_strategy,
    overnight_strategy, powerzones_strategy, crash_strategy, vol_panics_strategy,
    vxx_trend_strategy, new_highs_strategy, terror_gaps_strategy,
)

# Environment variables, Colab Secrets, or hidden interactive prompts.
# Do not replace these calls with literal secrets in saved cells.
API_KEY = read_notebook_secret("API_KEY")
API_SECRET = read_notebook_secret("API_SECRET")
client = StockHistoricalDataClient(API_KEY, API_SECRET)
del API_KEY, API_SECRET  # the SDK client retains its runtime credentials


### Symbols, dates, costs and execution defaults

Your backtest starts on `start_date`; earlier bars only initialize indicators. Supply a string such as `"SPY"` or a list such as `["SPY", "QQQ"]`. Each ticker gets its own initial capital. The matrix compares independently funded runs, not a shared-cash portfolio.

`position_steps` expresses fractions of a planned full position, e.g. `[0.5, 0.5]` for two equal units or `[0.1, 0.2, 0.3, 0.4]` for TPS. Source equal-unit systems buy equal **shares**, so later dollar notionals can differ. `position_fraction=0.95` reserves room for costs; it can be changed per call. Unsupported extra stages are rejected, not invented.

Short sections below explicitly assume borrow availability; set the annual `BORROW_RATE` to your estimate. IEX is not consolidated SIP and volume-based screens can differ. Split-adjusted research prices exclude dividends; dollar-price screens on adjusted data are an approximation. A custom ticker is a user-selected universe experiment, not an automatic SPX/VIX proxy.

In [ ]:
# Change these defaults once, or override symbols/dates beside any strategy below.
SYMBOLS = ["SPY"]             # "SPY" or ["SPY", "QQQ", "IWM"]
start_date = "2022-01-03"    # first date of the backtest (warm-up is fetched automatically)
end_date = "2024-12-31"      # inclusive last date; choose completed historical sessions
FEED = "iex"                # "sip" if your Alpaca entitlement permits it
INITIAL_CAPITAL = 100_000    # independent capital PER ticker
EXECUTION_MODE = "source_close"  # "next_open" is a timing variant
ASSUME_BORROW_AVAILABLE = True   # explicit research assumption for short sections
BORROW_RATE = 0.0                # annual decimal; choose a nonzero estimate if appropriate

research = NotebookResearch(
    client, cache=PROJECT / ".cache/alpaca_sdk", output=PROJECT / "reports/notebooks",
    feed=FEED, adjustment="split", initial_capital=INITIAL_CAPITAL,
    execution_mode=EXECUTION_MODE, fees=0.0005, slippage=0.0005,
    assume_borrow_available=ASSUME_BORROW_AVAILABLE, borrow_rate=BORROW_RATE,
)
plot_all_stocks_pnl_charts = research.plot_all_stocks_pnl_charts
results = {}  # collected runs for the performance matrix at the end


### Reading the outputs

Each run returns a `NotebookBatch` with `.results[symbol].portfolio` (vectorbt), `.figures[symbol]`, and `.performance` (DataFrame). Charts show completed winning/losing positions, P&L, daily equity and drawdown. Scale-ins count as one position. Open final positions affect equity but not closed-trade win rate. Saved HTML charts and CSV ledgers are under `reports/notebooks/`.

The performance matrix uses daily-return Sharpe (252 sessions/year, zero risk-free), calendar-time CAGR for runs at least one year, net episode return divided by entered notional, and two-way turnover. Avoid treating vectorbt's internal three-phase event frequency as daily returns; the matrix uses actual daily closes.

## 1. 2-Period RSI Under 5

**Source readiness:** ready. 

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** close > SMA200 AND RSI2 < 5
- **Additions:** none
- **Exit:** close > SMA5
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'The 2-Period RSI Under 5 on the S&P 500'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.rsi2'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=rsi_under_5_strategy,
)


## 2. Cumulative RSI — Two-Day Sum

**Source readiness:** experimental. Entry timing unstated; execution mode is explicit research assumption This sums today’s RSI(2) and yesterday’s RSI(2), not RSI(4).

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** close > SMA200 AND RSI2[t]+RSI2[t-1] < threshold
- **Additions:** none
- **Exit:** RSI2 > 65
- **Default parameters:** `{"threshold": 35}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Cumulative RSI — Two-Day Variants'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.cumulative'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=cumulative_rsi_strategy,
    allow_experimental=True,  # explicit assumptions explained above
)


### Named variants — optional comparison

Each item below preserves its documented thresholds and scale-in rules. Set `RUN_VARIANTS = True` to run them on the same dates. Keep parameter exploration in training/validation data; do not choose the best variant on a final test period.

- `cumulative50`: `{"threshold": 50}`

In [ ]:
RUN_VARIANTS = False
if RUN_VARIANTS:
    # Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
    strategy_name = 'Cumulative RSI — Two-Day Variants'
    symbols = SYMBOLS  # accepts a single ticker string or a list
    strategy_start_date = start_date
    strategy_end_date = end_date
    
    results['cumulative50'] = plot_all_stocks_pnl_charts(
        symbols, strategy_start_date, strategy_end_date,
        strategy_name=strategy_name,
        position_type=PositionType.LONG,
        position_steps=[1.0],
        strategy_func=cumulative_rsi_strategy,
        allow_experimental=True,  # explicit assumptions explained above
        variant='cumulative50',
    )


## 3. Double 7’s

**Source readiness:** ready. 

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** close > SMA200 AND close == rolling_min(close,7)
- **Additions:** none
- **Exit:** close == rolling_max(close,7)
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Double 7’s Strategy'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.double7'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=double_seven_strategy,
)


## 4. VIX RSI

**Source readiness:** ready. Requires supplied auxiliary CSV; no automatic proxy

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** SPY close > SMA200 AND VIX RSI2 > 90 AND VIX open > prior VIX close AND SPY RSI2 < 30
- **Additions:** none
- **Exit:** SPY RSI2 > 65
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

### Supply the auxiliary index history

Alpaca stock bars do not supply this index. Set the CSV path below to actual VIX or NYSE TRIN data with a `date` index and `close` (plus `open` for VIX). Include the warm-up period and exact market sessions. No VXX-for-VIX substitution occurs. Leave `None` to see an explicit skipped row and continue running the book.

In [ ]:
auxiliary_csv = None  # e.g. str(PROJECT / "data/vix.csv")


In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'VIX RSI Strategy'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.vix'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=vix_rsi_strategy,
    auxiliary_csv=auxiliary_csv,
)


## 5. TRIN

**Source readiness:** ready. Requires supplied auxiliary CSV; no automatic proxy

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** SPY close > SMA200 AND SPY RSI2 < 50 AND three consecutive TRIN close > 1
- **Additions:** none
- **Exit:** SPY RSI2 > 65
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

### Supply the auxiliary index history

Alpaca stock bars do not supply this index. Set the CSV path below to actual VIX or NYSE TRIN data with a `date` index and `close` (plus `open` for VIX). Include the warm-up period and exact market sessions. No VXX-for-VIX substitution occurs. Leave `None` to see an explicit skipped row and continue running the book.

In [ ]:
auxiliary_csv = None  # e.g. str(PROJECT / "data/vix.csv")


In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'TRIN Strategy'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.trin'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=trin_strategy,
    auxiliary_csv=auxiliary_csv,
)


## 6. S&P Short

**Source readiness:** ready. 

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Short rules

- **Entry:** close < SMA200 AND at least four consecutive positive close changes
- **Additions:** none
- **Exit:** close < SMA5
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'S&P Short Strategy'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.sp_short'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.SHORT,
    position_steps=[1.0],
    strategy_func=sp_short_strategy,
)


## 7. Ten-Day Closing Low — Five-Day Hold

**Source readiness:** ready. 

**Universe:** Stocks; historical point-in-time universe and delisted membership must be supplied

### Long rules

- **Entry:** close == rolling_min(close,10) AND close > SMA200 AND mean(volume[t-100:t-1]) >= 250000 AND close > 5
- **Additions:** none
- **Exit:** five sessions after entry, close; next-open variant described in methodology
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Ten-Day Closing Low — Five-Day Hold'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.ten_day'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=ten_day_low_strategy,
)


## 8. Overnight SPY Holding Test

**Source readiness:** ready. 

**Universe:** SPY / equity ETFs or explicitly documented market application; SPX not assumed tradable

### Long rules

- **Entry:** every session close
- **Additions:** none
- **Exit:** next session open
- **Default parameters:** `{}`

All “above/below” thresholds are strict; extrema ties qualify. Exits close the whole accumulated position.

In [ ]:
# Inputs for this strategy — edit ticker(s), dates, direction or allocation here.
strategy_name = 'Overnight SPY Holding Test'
symbols = SYMBOLS  # accepts a single ticker string or a list
strategy_start_date = start_date
strategy_end_date = end_date

results['short_term.overnight'] = plot_all_stocks_pnl_charts(
    symbols, strategy_start_date, strategy_end_date,
    strategy_name=strategy_name,
    position_type=PositionType.LONG,
    position_steps=[1.0],
    strategy_func=overnight_strategy,
    execution_mode="source_close",  # entry close, exit next open
)


## Book performance matrix

One row per strategy, direction and ticker. Skipped rows retain their reason. Rerunning a cell replaces that result in the in-memory comparison; every attempted configuration is still journaled on disk. This table is research output, not a recommendation.

In [ ]:
matrix = performance_matrix(results)
display_performance_matrix(matrix)
output_file = PROJECT / "reports/notebooks" / "short_term_trading_strategies_performance.csv"
output_file.parent.mkdir(parents=True, exist_ok=True)
matrix.to_csv(output_file, index=False)
print("Saved performance matrix:", output_file)


## Inspect orders, trades and a vectorbt portfolio

Choose any successful strategy key and ticker from `results`. The returned objects remain available for additional vectorbt analysis. Fees include settled borrow charges; reported equity also deducts outstanding borrow liabilities.

In [ ]:
# Example (uncomment after a successful run):
# batch = next(batch for batch in results.values() if batch.results)
# ticker = next(iter(batch.results))
# portfolio = batch.results[ticker].portfolio
# display(batch.results[ticker].orders)
# display(portfolio.positions.records_readable)
# portfolio.positions.regroup(False)[ticker].closed.plot_pnl().show()


## Research limitations

Missing/duplicate sessions fail validation rather than being filled. VIX/TRIN need auxiliary data; incomplete source specifications remain blocked. Current ticker lists can have survivorship bias; VXX has historical product differences. No raw-price corporate-action ledger, live execution, historical locates, margin liquidation or automatic stop is added. Read `docs/methodology.md` and the source catalog for precise assumptions. Authenticated Alpaca and hosted Colab must be verified with your credentials; local notebook tests use mocked SDK responses, never fabricated market results.

### Other mentions without executable rules

Cumulative RSI45 conflicts between RSI(3) over two days and RSI(2) over three days. Stock-specific Cumulative RSI omits necessary windows; VIX Stretches has conflicting exit equality; End of Month lacks a mandatory entry date. These are not runnable strategies and are listed as blocked in the catalog.